# IndicCorpV2 — monolingual candidate collection

Purpose: stream selected IndicCorpV2 language subsets to create *candidate-only* files for language-only categories. This does not create any language-English mixed-script data. The corpus is primarily web/news/book text, so domain review is mandatory before voice-agent training.

Source: https://huggingface.co/datasets/ai4bharat/IndicCorpV2

The dataset card reports CC-0 for its created data, but re-check current license and any upstream restrictions. Streaming is used to avoid downloading the full corpus.

In [ ]:
!pip -q install datasets pandas

In [ ]:
from datasets import load_dataset
from pathlib import Path
import pandas as pd, re, json, hashlib

OUT = Path('/content/data/external/indiccorp_v2')
OUT.mkdir(parents=True, exist_ok=True)
# Update paths only if the current dataset card lists different subset directories.
LANGS = {
  'English-only': {'label':'en-IN','data_dir':'data/eng_Latn','script':None},
  'Hindi-only Devanagari': {'label':'hi-IN','data_dir':'data/hin_Deva','script':r'[\u0900-\u097F]'},
  'Punjabi-only': {'label':'pa-IN','data_dir':'data/pan_Guru','script':r'[\u0A00-\u0A7F]'},
  'Bengali-only': {'label':'bn-IN','data_dir':'data/ben_Beng','script':r'[\u0980-\u09FF]'},
  'Gujarati-only': {'label':'gu-IN','data_dir':'data/guj_Gujr','script':r'[\u0A80-\u0AFF]'},
  'Kannada-only': {'label':'kn-IN','data_dir':'data/kan_Knda','script':r'[\u0C80-\u0CFF]'},
  'Malayalam-only': {'label':'ml-IN','data_dir':'data/mal_Mlym','script':r'[\u0D00-\u0D7F]'},
  'Marathi-only': {'label':'mr-IN','data_dir':'data/mar_Deva','script':r'[\u0900-\u097F]'},
  'Telugu-only': {'label':'te-IN','data_dir':'data/tel_Telu','script':r'[\u0C00-\u0C7F]'},
  'Assamese-only': {'label':'as-IN','data_dir':'data/asm_Beng','script':r'[\u0980-\u09FF]'},
  'Tamil-only': {'label':'ta-IN','data_dir':'data/tam_Taml','script':r'[\u0B80-\u0BFF]'},
}
TARGET_PER_CATEGORY = 20000
print('Categories:', len(LANGS), 'target candidate rows:', len(LANGS)*TARGET_PER_CATEGORY)


## Stream each subset and save candidate rows
If a subset path/config fails, stop and inspect the current dataset card rather than silently substituting another source. Script checks cannot distinguish languages that share or overlap Unicode blocks (especially Bengali/Assamese), so source subset identity and review are still required.

In [ ]:
summary=[]
for category, cfg in LANGS.items():
    print('\nCollecting:', category, cfg['data_dir'])
    stream = load_dataset('ai4bharat/IndicCorpV2', 'indiccorp_v2', data_dir=cfg['data_dir'], split='train', streaming=True)
    rows=[]; seen=set(); scanned=0
    for item in stream:
        scanned += 1
        text = str(item.get('text','')).strip()
        if not text: continue
        if cfg['script'] and not re.search(cfg['script'], text): continue
        # Language-only candidate gate: require native script for Indic categories.
        # For English, reject rows containing common Indic script blocks; review still required.
        if category == 'English-only' and re.search(r'[\u0900-\u0DFF]', text): continue
        norm = re.sub(r'\s+',' ',text.casefold()).strip()
        if norm in seen: continue
        seen.add(norm)
        row_id = hashlib.sha256((category+'|'+str(scanned)+'|'+norm).encode('utf-8')).hexdigest()[:20]
        rows.append({'text':text,'label':cfg['label'],'language_category':category,'source_dataset':'AI4Bharat IndicCorpV2','source_url':'https://huggingface.co/datasets/ai4bharat/IndicCorpV2','source_row_id':row_id,'group_id':row_id,'source_split':'train_stream','is_synthetic':False,'review_status':'pending_language_and_domain_review','license_status':'verify_current_CC0_metadata'})
        if len(rows) >= TARGET_PER_CATEGORY: break
    path = OUT / (category.lower().replace(' ','_').replace('-','_')+'_candidates.csv')
    pd.DataFrame(rows).to_csv(path,index=False)
    summary.append({'category':category,'label':cfg['label'],'rows_collected':len(rows),'rows_scanned':scanned,'target':TARGET_PER_CATEGORY,'status':'candidate_only_not_approved','path':str(path)})
    print('Saved',len(rows),'from',scanned,'scanned to',path)
summary_df=pd.DataFrame(summary)
display(summary_df)
summary_path=OUT/'indiccorp_v2_collection_summary.csv'
summary_df.to_csv(summary_path,index=False)
print('Summary:',summary_path)


## Download candidate files
Only run after the collection cell finishes. These are candidate pools, not final train/validation/test splits.

In [ ]:
from google.colab import files
files.download(str(OUT/'indiccorp_v2_collection_summary.csv'))
# Zip category CSVs so they can be transferred as one artifact.
import zipfile
zip_path=OUT/'indiccorp_v2_candidates.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*_candidates.csv'): z.write(p,arcname=p.name)
files.download(str(zip_path))
